Import packages

In [473]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import StandardScaler, MinMaxScaler, Normalizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
import joblib

## Recommendation System and NearestNeighbors, Cosine_similarity

### Load the Dataset

### Orginal Dataframe without Encoding:

In [474]:
global_mobile_review_df = pd.read_csv(r"C:\Users\jagad\Documents\my_classes\Tasks\mini_project_4-global_mobile_reviews\data\processed_data\Mobile_Reviews_Sentiment_before_encoding.csv")


#### Feature selection:


In [475]:
product_df = (
            global_mobile_review_df.groupby(["brand", "model"], as_index=False)
            .agg({
                "price_usd": "mean",
                "rating": "mean",
                "battery_life_rating": "mean",
                "camera_rating": "mean",
                "performance_rating": "mean",
                "design_rating": "mean",
                "display_rating": "mean"
            })
        )

product_df = product_df.round(2)
print(product_df.shape)

joblib.dump(product_df, r"C:\Users\jagad\Documents\my_classes\Tasks\mini_project_4-global_mobile_reviews\models\product_df.pkl")
print(" product_df is saved as pkl file")

(22, 9)
 product_df is saved as pkl file


#### One Hot Encoding the brand:

In [476]:
product_encoded = pd.get_dummies(
                                    product_df,
                                    columns=["brand"],
                                    dtype=int
)
product_encoded.head()

,model,price_usd,rating,battery_life_rating,camera_rating,performance_rating,design_rating,display_rating,brand_Apple,brand_Google,brand_Motorola,brand_OnePlus,brand_Realme,brand_Samsung,brand_Xiaomi
0,iPhone 13,1108.26,3.13,2.75,2.75,2.77,2.77,2.71,1,0,0,0,0,0,0
1,iPhone 14,1099.26,3.15,2.74,2.78,2.72,2.77,2.76,1,0,0,0,0,0,0
2,iPhone 15 Pro,1103.40,3.06,2.68,2.71,2.66,2.68,2.69,1,0,0,0,0,0,0
3,iPhone SE,1103.39,3.08,2.73,2.70,2.73,2.69,2.74,1,0,0,0,0,0,0
4,Pixel 6,807.92,3.09,2.71,2.71,2.69,2.71,2.72,0,1,0,0,0,0,0


In [483]:
X = product_encoded.drop(columns=["model"])

print(product_encoded.shape)
print(X.columns)

(22, 15)
Index(['price_usd', 'rating', 'battery_life_rating', 'camera_rating',
       'performance_rating', 'design_rating', 'display_rating', 'brand_Apple',
       'brand_Google', 'brand_Motorola', 'brand_OnePlus', 'brand_Realme',
       'brand_Samsung', 'brand_Xiaomi'],
      dtype='str')


Scaling the features:

In [484]:
# # Using StandardScalar:
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(X)

# Using MinmaxScalar:
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# # Using Normalizer:
# scaler = Normalizer()
# X_scaled = scaler.fit_transform(X)


joblib.dump(scaler, r"C:\Users\jagad\Documents\my_classes\Tasks\mini_project_4-global_mobile_reviews\models\scaler.pkl" )
print("Scalar is saved as pkl file")

print(X_scaled.shape)

joblib.dump(X_scaled, r"C:\Users\jagad\Documents\my_classes\Tasks\mini_project_4-global_mobile_reviews\models\X_scaled.pkl")
print("X_scaled is saved as pkl file")


Scalar is saved as pkl file
(22, 14)
X_scaled is saved as pkl file


#### Train the Model:

##### Using NearestNeighbors:

In [479]:
# Train the model:
recommend_model = NearestNeighbors( n_neighbors=6,
                                    metric="cosine",
                                    algorithm="brute"
)
recommend_model.fit(X_scaled)
print(recommend_model)


# save the trained model in .pkl fromat:
joblib.dump(recommend_model, r"C:\Users\jagad\Documents\my_classes\Tasks\mini_project_4-global_mobile_reviews\models\recommend_model.pkl")
print(" recommend model is saved as pkl file")

NearestNeighbors(algorithm='brute', metric='cosine', n_neighbors=6)
 recommend model is saved as pkl file


### Create Recommendation Function:

#### Based on NearestNeighbors:

In [480]:
def recommend_mobile( brand, price, rating, battery, camera, performance, 
                        design,  display, top_n):

    # Create user input
    user_input = {
                    "price_usd": price,
                    "rating": rating,
                    "battery_life_rating": battery,
                    "camera_rating": camera,
                    "performance_rating": performance,
                    "design_rating": design,
                    "display_rating": display,

                    "brand_Apple": 0,
                    "brand_Google": 0,
                    "brand_Motorola": 0,
                    "brand_OnePlus": 0,
                    "brand_Realme": 0,
                    "brand_Samsung": 0,
                    "brand_Xiaomi": 0
    }

    user_input[f"brand_{brand}"] = 1

    user_df = pd.DataFrame([user_input])

    user_scaled = scaler.transform(user_df)

    distances, indices = recommend_model.kneighbors( user_scaled, n_neighbors=top_n )

    recommendations = product_df.iloc[indices[0]].copy()

    recommendations["Similarity (%)"] = ( (1 - distances[0]) * 100 ).round(2)

    return recommendations[
        [
                            "brand",
                            "model",
                            "price_usd",
                            "rating",
                            "battery_life_rating",
                            "camera_rating",
                            "performance_rating",
                            "design_rating",
                            "display_rating",
                            "Similarity (%)"
        ]
    ]

#### Test

In [481]:
recommend_mobile( brand="Apple", price=500, rating=3.10, battery=2.73, camera=2.74, 
                    performance=2.72, design=2.71, display=2.74, top_n=5 )

,brand,model,price_usd,rating,battery_life_rating,camera_rating,performance_rating,design_rating,display_rating,Similarity (%)
1,Apple,iPhone 14,1099.26,3.15,2.74,2.78,2.72,2.77,2.76,91.65
0,Apple,iPhone 13,1108.26,3.13,2.75,2.75,2.77,2.77,2.71,89.33
3,Apple,iPhone SE,1103.39,3.08,2.73,2.70,2.73,2.69,2.74,87.80
14,Realme,Realme Narzo 70,392.71,3.11,2.74,2.73,2.75,2.71,2.73,70.98
16,Samsung,Galaxy Note 20,899.73,3.13,2.74,2.73,2.76,2.75,2.73,70.48


In [482]:
print(global_mobile_review_df["rating"].describe())


count    50000.000000
mean         3.102160
std          1.248661
min          1.000000
25%          2.000000
50%          3.000000
75%          4.000000
max          5.000000
Name: rating, dtype: float64
